In [2]:
import pandas as pd
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# --------------------------------
# 1. Load Dataset
# --------------------------------

df = pd.read_csv("IMDB_Dataset_CLEANED.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())

# --------------------------------
# 2. Missing Values
# --------------------------------

print("\nMissing Values:")
print(df.isnull().sum())

df = df.dropna(subset=['review'])

# --------------------------------
# 3. Duplicate Reviews
# --------------------------------

print("\nDuplicate Rows:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicates After Removal:", df.duplicated().sum())

# --------------------------------
# 4. Store Original Review
# --------------------------------

df['original_review'] = df['review']

# --------------------------------
# 5. Text Cleaning
# --------------------------------

def clean_text(text):
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Convert lowercase
    text = text.lower()
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )
    
    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['cleaned_review'] = df['review'].apply(clean_text)

# --------------------------------
# 6. Stopword Removal
# --------------------------------

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    
    words = word_tokenize(text)
    
    filtered_words = [
        word for word in words
        if word not in stop_words
    ]
    
    return ' '.join(filtered_words)

df['cleaned_review'] = df['cleaned_review'].apply(
    remove_stopwords
)

# --------------------------------
# 7. Stemming
# --------------------------------

stemmer = PorterStemmer()

def stem_text(text):
    
    words = word_tokenize(text)
    
    stemmed_words = [
        stemmer.stem(word)
        for word in words
    ]
    
    return ' '.join(stemmed_words)

df['cleaned_review'] = df['cleaned_review'].apply(
    stem_text
)

# --------------------------------
# 8. Compare Original vs Cleaned
# --------------------------------

print("\nOriginal vs Cleaned Reviews:\n")

for i in range(5):
    
    print("ORIGINAL:")
    print(df['original_review'].iloc[i])
    
    print("\nCLEANED:")
    print(df['cleaned_review'].iloc[i])
    
    print("\nSENTIMENT:")
    print(df['sentiment'].iloc[i])
    
    print("=" * 80)

# --------------------------------
# 9. Create Final Dataset
# --------------------------------

final_df = df[
    ['original_review', 'sentiment', 'cleaned_review']
]

# --------------------------------
# 10. Export CSV
# --------------------------------

final_df.to_csv(
    "preprocessed_imdb_reviews.csv",
    index=False
)

print("\nPreprocessing completed successfully!")
print("Final Dataset Shape:", final_df.shape)
print("Saved as: preprocessed_imdb_reviews.csv")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Harshini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Harshini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Harshini\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset Shape: (49396, 2)

First 5 Records:


,review,sentiment
0,"$25,000 Pyramid Clues: Deep Blue Sea. Tremors....",negative
1,0.5/10. This movie has absolutely nothing good...,negative
2,"0*'s Christian Slater, Tara Reid, Stephen Dorf...",negative
3,"102 Dalmatians (2000, Dir. Kevin Lima) <br /><...",negative
4,102 DALMATIANS [Walt Disney]: I wasn't a fan o...,negative



Missing Values:
review       0
sentiment    0
dtype: int64

Duplicate Rows: 0
Duplicates After Removal: 0

Original vs Cleaned Reviews:

ORIGINAL:
$25,000 Pyramid Clues: Deep Blue Sea. Tremors. Slither. Eight Legged Freaks.<br /><br />Pyramid Category: Movies that were funnier and more thrilling than Snakes on a Plane.<br /><br />Hell, with that definition I'd have to include the relatively harrowing journey of Ted and Elaine in Airplane! as superior to Snakes in both laughs and thrills.<br /><br />The sad truth is that this isn't even close to the mother of all unintentionally intentional funny snake movies: Anaconda! Besides the never to be seen again casting of JLo-Cube-O.Wilson-Stoltz-Wuhrer in the same flick, you had Jon Voight pulling off the all-time cinematic heist. His final scene alone represents everything SOAP tried and failed to do as a "so-ludicrous-it's-fun" movie.<br /><br />In the end, Snakes on a Plane is definitive proof that studio execs and fanboys make the worst 